# 03 — Baselines (Adim 5)

3 profit baseline + 3 route baseline. Sonuclar `data/processed/baseline_results.json`'a yazilir.

In [ ]:
from __future__ import annotations
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print('repo root:', REPO_ROOT)

SEED = 42

In [ ]:
from ml.src.features import M1_FEATURES, BOOLEAN_FEATURES
from ml.src.baselines import (
    rule_based_predict, make_linear_regressor, make_logistic_classifier,
    make_rf_regressor, make_rf_classifier, stratified_split,
    evaluate_regression, evaluate_classification, majority_baseline_predict,
)

df = pd.read_parquet(REPO_ROOT / 'data/processed/training_set_v1.parquet')
pipe = joblib.load(REPO_ROOT / 'models/feature_pipeline.pkl')
print('df shape:', df.shape)

seen = set(); feat_cols = []
for c in M1_FEATURES + BOOLEAN_FEATURES:
    if c in seen or c not in df.columns: continue
    seen.add(c); feat_cols.append(c)
print('feat_cols:', len(feat_cols))

In [ ]:
train, val, test = stratified_split(
    df, 'expected_profit_try',
    stratify_cols=['raw_material', 'processing_route_candidate'],
    seed=SEED,
)
print(f'train={len(train)}  val={len(val)}  test={len(test)}')

X_train = pipe.transform(train[feat_cols])
X_val = pipe.transform(val[feat_cols])
X_test = pipe.transform(test[feat_cols])
y_train_p = train['expected_profit_try'].values
y_val_p = val['expected_profit_try'].values
y_test_p = test['expected_profit_try'].values
y_train_r = train['recommended_route_label'].values
y_val_r = val['recommended_route_label'].values
y_test_r = test['recommended_route_label'].values

## Profit baselines

In [ ]:
# 1) Rule-based
y_pred_rule = rule_based_predict(test)
m_rule = evaluate_regression(y_test_p, y_pred_rule)
print('rule_based:', m_rule)

# 2) LinearRegression
lr = make_linear_regressor()
lr.fit(X_train, y_train_p)
y_pred_lr = lr.predict(X_test)
m_lr = evaluate_regression(y_test_p, y_pred_lr)
print('linear_regression:', m_lr)

# 3) RandomForestRegressor
rf = make_rf_regressor()
rf.fit(X_train, y_train_p)
y_pred_rf = rf.predict(X_test)
m_rf = evaluate_regression(y_test_p, y_pred_rf)
print('random_forest:', m_rf)

## Route baselines

In [ ]:
# 1) Majority
y_pred_maj = majority_baseline_predict(y_train_r, len(y_test_r))
m_majority = evaluate_classification(y_test_r, y_pred_maj)
print('majority:', m_majority)

# 2) LogisticRegression
logr = make_logistic_classifier()
logr.fit(X_train, y_train_r)
y_pred_log = logr.predict(X_test)
y_proba_log = logr.predict_proba(X_test)
m_logr = evaluate_classification(y_test_r, y_pred_log, y_proba_log, list(logr.classes_))
print('logistic:', m_logr)

# 3) RandomForestClassifier
rfc = make_rf_classifier()
rfc.fit(X_train, y_train_r)
y_pred_rfc = rfc.predict(X_test)
y_proba_rfc = rfc.predict_proba(X_test)
m_rfc = evaluate_classification(y_test_r, y_pred_rfc, y_proba_rfc, list(rfc.classes_))
print('random_forest:', m_rfc)

In [ ]:
# Save baseline_results.json
results = {
    'profit': {
        'rule_based': m_rule,
        'linear_regression': m_lr,
        'random_forest': m_rf,
    },
    'route': {
        'majority_baseline': m_majority,
        'logistic_regression': m_logr,
        'random_forest': m_rfc,
    },
    'split': {
        'train': len(train), 'val': len(val), 'test': len(test),
        'seed': SEED,
        'stratify_cols': ['raw_material', 'processing_route_candidate'],
    },
}
out_path = REPO_ROOT / 'data/processed/baseline_results.json'
out_path.write_text(json.dumps(results, indent=2, default=float))
print('saved:', out_path)
results

## Confusion matrix (RF route)

In [ ]:
from sklearn.metrics import confusion_matrix
labels = list(rfc.classes_)
cm = confusion_matrix(y_test_r, y_pred_rfc, labels=labels)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=75, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('RF route classifier — confusion matrix')
plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: profit RMSE comparison
names = ['rule_based','linear_regression','random_forest']
rmses = [m_rule['rmse'], m_lr['rmse'], m_rf['rmse']]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(names, rmses, color=['#1f77b4','#ff7f0e','#2ca02c'])
ax.set_yscale('log')
ax.set_ylabel('RMSE (TRY, log)')
ax.set_title('Profit baseline RMSE comparison')
plt.tight_layout()
plt.show()

# Bar chart: route accuracy
names = ['majority','logistic','random_forest']
accs = [m_majority['accuracy'], m_logr['accuracy'], m_rfc['accuracy']]
f1s = [m_majority['macro_f1'], m_logr['macro_f1'], m_rfc['macro_f1']]
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(names))
ax.bar(x-0.2, accs, width=0.4, label='accuracy', color='#1f77b4')
ax.bar(x+0.2, f1s, width=0.4, label='macro_f1', color='#ff7f0e')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0, 1.05); ax.legend()
ax.set_title('Route baseline accuracy/macro_f1')
plt.tight_layout()
plt.show()